In [0]:
spark.sql(
    "DROP TABLE IF EXISTS workspace.gold.dim_location"
)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.dim_location (
    location_key BIGINT,
    place STRING,
    latitude DOUBLE,
    longitude DOUBLE
)
""")

In [0]:
from pyspark.sql import functions as F

location = (
    spark.table(
        "workspace.silver.usgs_earthquakes"
    )
    .select(
        "place",
        "latitude",
        "longitude"
    )
    .dropDuplicates()
    .withColumn(
        "location_key",
        F.xxhash64(
            "place",
            "latitude",
            "longitude"
        )
    )
)

location = location.select(
    "location_key",
    "place",
    "latitude",
    "longitude"
)

In [0]:
location.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable("workspace.gold.dim_location")